In [2]:
import io
import os
import sqlite3
from enum import Enum
from logging import DEBUG, basicConfig, getLogger
from pathlib import Path
from typing import Any, Optional

import chess
import chess.engine
import chess.pgn
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping

In [3]:
# Set memory growth to avoid allocating all GPU memory at once
physical_devices = tf.config.list_physical_devices("GPU")
if physical_devices:
    print(f"Found {len(physical_devices)} GPU(s)")
    for device in physical_devices:
        tf.config.experimental.set_memory_growth(device, True)
        print(f"Memory growth set to True for {device}")
else:
    print("No GPU found, using CPU")

Found 1 GPU(s)
Memory growth set to True for PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')


In [4]:
# Use mixed precision to reduce memory usage.
try:
    policy = tf.keras.mixed_precision.Policy("mixed_float16")
    tf.keras.mixed_precision.set_global_policy(policy)
    print("Using mixed precision policy")
except:
    print("Mixed precision not supported or enabled")

Using mixed precision policy


In [ ]:
basicConfig(filename="eval.log", level=DEBUG, format="%(asctime)s:%(filename)s:%(funcName)s:[%(levelname)s]: %(message)s", force=True, filemode="w")
logger = getLogger(__name__)

In [ ]:
import concurrent
import contextlib

from tqdm import tqdm


class Errors(Enum):
    NONE = 0
    INACCURACY = 1
    MISTAKE = 2
    BLUNDER = 3

    @property
    def threshold(self) -> float:
        thresh = {
            Errors.BLUNDER: 0.3,
            Errors.MISTAKE: 0.2,
            Errors.INACCURACY: 0.1,
        }
        return thresh[self]

class ChessAnalyzer:
    MIN_PLY_COUNT = 5
    RAPID_THRESH = 1499


    def __init__(self, db_path: Path, sf_path: Path="stockfish", depth: int=18, threads: int=4, max_workers: int|None = None):
        """Initialize the chess analyzer.

        Args:
            db_path (str): Path to the SQLite database
            stockfish_path (str): Path to the Stockfish executable
            depth (int): Analysis depth for Stockfish
            threads (int): Number of threads for Stockfish to use
            max_workers (int): Maximum number of parallel workers
        """
        self.db_path = db_path
        self.stockfish_path = sf_path
        self.depth = depth
        self.engine_threads = threads
        self.max_workers = max_workers or os.cpu_count()

        self.conn = sqlite3.connect(self.db_path)

        try:
            engine = chess.engine.SimpleEngine.popen_uci(sf_path)
            engine.quit()
            logger.debug(f"Stockfish found at {sf_path}")
        except Exception:
            logger.exception("Error initializing Stockfish")
            logger.debug("Please provide a valid path to Stockfish executable")
            raise

        self._init_analysis_tables()

    def _init_analysis_tables(self) -> None:
        """Create the analysis tables if they don't exist."""
        conn = self.conn
        cursor = conn.cursor()

        # Game level evaluations table
        cursor.execute("""
        CREATE TABLE IF NOT EXISTS evaluation_game_level (
            game_id INTEGER PRIMARY KEY,
            time_control TEXT,          -- Time control of the game
            estimated_time INTEGER,     -- Estimated game time in seconds
            game_type TEXT,             -- Game type (rapid, blitz, classical, etc.)
            black_blunders INTEGER,     -- Number of blunders made by black
            black_mistakes INTEGER,     -- Number of mistakes made by black
            black_inaccuracies INTEGER, -- Number of inaccuracies made by black
            white_blunders INTEGER,     -- Number of blunders made by white
            white_mistakes INTEGER,     -- Number of mistakes made by white
            white_inaccuracies INTEGER, -- Number of inaccuracies made by white
            analysis_completed TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
        """)

        # Move level evaluations table
        cursor.execute("""
        CREATE TABLE IF NOT EXISTS evaluation_move_level (
            game_id INTEGER,
            halfmove_count INTEGER,
            move TEXT,
            fen TEXT,
            turn INTEGER,               -- 0 for white, 1 for black
            error INTEGER,              -- 0: none, 1: inaccuracy, 2: mistake, 3: blunder
            cp INTEGER,                 -- Centipawn evaluation
            mate INTEGER,               -- Mate in X moves (NULL if no mate)
            time_ratio REAL,            -- Ratio of time spent on this move
            winning_chance REAL,        -- Probability of winning
            drawing_chance REAL,        -- Probability of drawing
            losing_chance REAL,         -- Probability of losing
            is_check INTEGER,           -- 1 if check, 0 otherwise
            is_checkmate INTEGER,       -- 1 if checkmate, 0 otherwise
            PRIMARY KEY (game_id, halfmove_count),
            FOREIGN KEY (game_id) REFERENCES evaluation_game_level(game_id)
        )
        """)

        conn.commit()

    def evaluate_move(self, wdl_before: chess.engine.Wdl, wdl_after: chess.engine.Wdl) -> Errors:

        before_expected_score = wdl_before[0] + (0.5 * wdl_before[1])
        after_expected_score = wdl_after[2] + (0.5 * wdl_after[1])

        score_drop = (before_expected_score - after_expected_score) / 1000

        if score_drop >= Errors.BLUNDER.threshold:
            return Errors.BLUNDER
        if score_drop >= Errors.MISTAKE.threshold:
            return Errors.MISTAKE
        if score_drop >= Errors.INACCURACY.threshold:
            return Errors.INACCURACY

        return None

    def parse_time_control(self, time_control_str: str) -> tuple:
        """Parse the time control string and calculate estimated game time.

        Args:
            time_control_str (str): Time control string (e.g., "180+0", "300+2")

        Returns:
            tuple: (base_time, increment, estimated_time)
        """
        if not time_control_str or time_control_str == "-":
            return (None, None, None)

        try:
            # Handle standard time control format "base+increment"
            if "+" in time_control_str:
                parts = time_control_str.split("+")
                base_time = int(parts[0])
                increment = int(parts[1])

                # Calculate estimated time: base_time + (40 * increment)
                estimated_time = base_time + (40 * increment)

                return (base_time, increment, estimated_time)
            # Handle time formats without increment
            base_time = int(time_control_str)
        except Exception:
            logger.exception(f"Error parsing time control '{time_control_str}'")
            return (None, None, None)
        else:
            return (base_time, 0, base_time)

    def encode_move(self, move_obj: chess.Move) -> np.array:
        """Encode a python-chess move to a normalized 5D vector.

        [from_col, from_row, to_col, to_row, promotion]
        All values are scaled to [0, 1] for Transformer compatibility.
        """
        from_sq = move_obj.from_square
        to_sq = move_obj.to_square

        from_col = chess.square_file(from_sq) / 7.0
        from_row = chess.square_rank(from_sq) / 7.0
        to_col = chess.square_file(to_sq) / 7.0
        to_row = chess.square_rank(to_sq) / 7.0

        # Normalize promotion: 0 if not a promotion, else map [1, 2, 3, 4, 5] → [0.2, ..., 1.0]
        # Promotion types: None (0), Knight (2), Bishop (3), Rook (4), Queen (5)
        promotion_raw = move_obj.promotion if move_obj.promotion else 0
        promotion_normalized = promotion_raw / 5.0  # Max promotion code is 5 (Queen)

        return np.array([from_col, from_row, to_col, to_row, promotion_normalized], dtype=np.float32)

    def encode_fen(self, fen: str) -> np.array:
        """Encode FEN string to include.

        - Signed material values: white +ve, black -ve (normalized to [-1, 1])
        - Castling rights (4 binary flags)
        - En passant square (2 normalized floats)

        Output shape: (64 + 4 + 2) = (70,)
        """
        parts = fen.split(" ")
        board_fen = parts[0]
        castling = parts[2] if len(parts) > 2 else "-"
        en_passant = parts[3] if len(parts) > 3 else "-"

        # --- 1. Encode signed board pieces ---
        base_map = {
            "p": -1, "n": -2, "b": -3, "r": -4, "q": -5, "k": -6,
            "P": 1,  "N": 2,  "B": 3,  "R": 4,  "Q": 5,  "K": 6,
        }
        expanded = ""
        for ch in board_fen:
            if ch.isdigit():
                expanded += " " * int(ch)
            elif ch == "/":
                continue
            else:
                expanded += ch

        # Map to values in [-1, 1] (divide by max absolute value 6)
        board_encoded = [base_map.get(ch, 0) / 6.0 if ch != " " else 0.0 for ch in expanded]

        # --- 2. Castling rights: [K, Q, k, q]
        castling_flags = [
            1.0 if "K" in castling else 0.0,
            1.0 if "Q" in castling else 0.0,
            1.0 if "k" in castling else 0.0,
            1.0 if "q" in castling else 0.0,
        ]

        # --- 3. En passant square: normalized col and row
        if en_passant != "-" and len(en_passant) == 2:
            col = ord(en_passant[0]) - ord("a")  # 0-7
            row = int(en_passant[1]) - 1         # 0-7
            en_passant_encoded = [col / 7.0, row / 7.0]
        else:
            en_passant_encoded = [0.0, 0.0]

        return np.array(board_encoded + castling_flags + en_passant_encoded, dtype=np.float32)

    def analyze_game(self, game_data: tuple[str]) -> dict[Any]:
        """Analyze a single chess game using Stockfish.

        Args:
            game_data (tuple): Game data from the database

        Returns:
            dict: Analysis results
        """
        game_id = game_data[0]
        pgn_text = game_data[14]
        time_control = game_data[10]
        estimated_time = game_data[16]
        game_type = game_data[17]
        white_elo = game_data[6]
        black_elo = game_data[8]

        if "1/2" in game_data[9]:
            result = 0
        elif game_data[9].startswith("1"):
            result = 1
        elif game_data[9].startswith("0"):
            result = -1

        _, increment, _ = self.parse_time_control(time_control)

        if not pgn_text or pgn_text == "":
            logger.warning("No PGN data")
            return {
                "game_id": game_id,
                "error": "No PGN data",
                "time_control": time_control,
                "estimated_time": estimated_time,
                "game_type": game_type,
            }

        try:
            pgn_io = io.StringIO(pgn_text)
            game = chess.pgn.read_game(pgn_io)

            if game is None:
                logger.warning("Failed to parse PGN")
                return {
                    "game_id": game_id,
                    "error": "Failed to parse PGN",
                    "time_control": time_control,
                    "estimated_time": estimated_time,
                    "game_type": game_type,
                }

            if game.end().ply()/2 < self.MIN_PLY_COUNT:
                logger.warning("Game too short!")
                return {
                        "game_id": game_id,
                        "error": "Game too short",
                        "time_control": time_control,
                        "estimated_time": estimated_time,
                        "game_type": game_type,
                    }
        except Exception:
            logger.exception("Error parsing PGN")
            return {
                "game_id": game_id,
                "error": "Error parsing PGN",
                "time_control": time_control,
                "estimated_time": estimated_time,
                "game_type": game_type,
            }

        # Initialize Stockfish
        try:
            engine = chess.engine.SimpleEngine.popen_uci(self.stockfish_path)
            engine.configure({"Threads": self.engine_threads,
                              "Skill Level": 20,  # Max strength
                              "UCI_LimitStrength": False,
                              "UCI_ShowWDL": True,
            })
        except Exception:
            logger.exception("Error initializing engine.")
            return {
                "game_id": game_id,
                "error": "Error initializing engine",
                "time_control": time_control,
                "estimated_time": estimated_time,
                "game_type": game_type,
            }

        try:

            board = game.board()

            move_features = []

            white_blunders = 0
            white_mistakes = 0
            white_inaccuracies = 0
            black_blunders = 0
            black_mistakes = 0
            black_inaccuracies = 0

            curr_node = game
            prev_node = chess.pgn.Game()

            prev_prev_clock = None

            turn = int(chess.BLACK)

            halfmove_count = -1

            while curr_node:

                error = Errors.NONE
                move = curr_node.move

                if curr_node.parent is not None:
                    board.push(move)

                info = engine.analyse(board, chess.engine.Limit(depth=self.depth), info=chess.engine.Info.SCORE)

                score = info["score"].white()
                cp = score.score(mate_score=1000)
                mate = score.mate()
                time_spent = 0 if (not curr_node.parent) or (not prev_prev_clock) else prev_prev_clock - curr_node.clock()
                time_ratio = 0 if (not prev_node.parent) or (not prev_prev_clock) else time_spent / (prev_prev_clock + increment)
                halfmove_count += 1
                turn = int(not turn)
                is_check = board.is_check()
                is_checkmate = board.is_checkmate()

                try:
                    curr_node.wdl = info["wdl"]
                except KeyError:
                    wdl = chess.engine.Wdl(1000, 0, 0)
                    curr_node.wdl = chess.engine.PovWdl(wdl, not turn)

                if curr_node.parent:
                    error = self.evaluate_move(prev_node.wdl, curr_node.wdl)
                    if error is Errors.BLUNDER:
                        if turn is int(chess.BLACK):
                            white_blunders += 1
                        else:
                            black_blunders += 1
                    if error is Errors.MISTAKE:
                        if turn is int(chess.BLACK):
                            white_mistakes += 1
                        else:
                            black_mistakes += 1
                    if error is Errors.INACCURACY:
                        if turn is int(chess.BLACK):
                            white_inaccuracies += 1
                        else:
                            black_inaccuracies += 1

                features = {
                                "halfmove_count": halfmove_count,
                                "move": move,
                                "fen": board.fen(),
                                "turn": turn,
                                "error": error.value,
                                "cp": cp,
                                "mate": mate,
                                "time_ratio": time_ratio,
                                "winning_chance": curr_node.wdl.white().winning_chance(),
                                "drawing_chance": curr_node.wdl.white().drawing_chance(),
                                "losing_chance": curr_node.wdl.white().losing_chance(),
                                "is_check": int(is_check),
                                "is_checkmate": int(is_checkmate),
                             }

                move_features.append(features)

                prev_prev_clock, prev_node, curr_node = prev_node.clock(), curr_node, curr_node.next()

            # Close the engine
            engine.quit()

        except Exception:
            # Make sure to quit the engine if an error occurs
            with contextlib.suppress(Exception):
                engine.quit()

            logger.exception("Error analyzing game.")

            return {
                "game_id": game_id,
                "error": "Error analyzing game",
                "time_control": time_control,
                "estimated_time": estimated_time,
                "game_type": game_type,
            }
        else:
            return {
                "game_id": game_id,
                "time_control": time_control,
                "estimated_time": estimated_time,
                "game_type": game_type,
                "black_errors": (black_blunders, black_mistakes, black_inaccuracies),
                "white_errors": (white_blunders, white_mistakes, white_inaccuracies),
                "outcome": result,
                "moves": move_features,
                "target": (white_elo, black_elo),
                "total_move_count": halfmove_count,
            }

    def get_games_to_analyze(self, connection: sqlite3.Connection, limit: Optional[int] = None, game_type_filter: str = "rapid", offset: int = 0) -> sqlite3.Cursor:
        """Get games that haven't been analyzed yet, filtered by game type.

        Args:
            connection (sqlite3.Connection): Database connection
            limit (int, optional): Maximum number of games to retrieve
            game_type_filter (str): Type of games to filter for (rapid, blitz, classical, etc.)
            offset (int): Offset for pagination

        Returns:
            cursor: Cursor for the executed query.
        """
        cursor = connection.cursor()

        query = """
        SELECT g.* FROM game_with_type g
        LEFT JOIN evaluation_game_level a ON g.ID = a.game_id
        WHERE a.game_id IS NULL AND g.result IS NOT NULL
        """

        # Apply game type filter
        if game_type_filter:
            query += f" AND g.game_type = '{game_type_filter}' AND (g.MOVES != '1-0\n' AND g.MOVES != '0-1\n')"

        # Add ORDER BY to ensure consistent results when using offset
        query += " ORDER BY g.ID"

        # Apply limit if specified
        if limit:
            query += f" LIMIT {limit}"

        # Apply offset for resuming analysis
        if offset > 0:
            query += f" OFFSET {offset}"

        cursor.execute(query)
        return cursor

    def save_to_database(self, analysis_results: List[Dict[str, Any]]) -> None:
        """Save analysis results to the database.

        Args:
            analysis_results (List[Dict]): List of analysis results from analyzed games
        """
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()

        try:
            conn.execute("BEGIN TRANSACTION")

            for result in analysis_results:
                if "error" in result:
                    logger.warning(f"Skipping game {result['game_id']} due to error: {result['error']}")
                    continue

                # Insert game level data
                game_id = result["game_id"]
                black_blunders, black_mistakes, black_inaccuracies = result["black_errors"]
                white_blunders, white_mistakes, white_inaccuracies = result["white_errors"]

                cursor.execute("""
                INSERT INTO evaluation_game_level 
                (game_id, time_control, estimated_time, game_type, 
                black_blunders, black_mistakes, black_inaccuracies,
                white_blunders, white_mistakes, white_inaccuracies)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                """, (
                    game_id, result["time_control"], result["estimated_time"], result["game_type"],
                    black_blunders, black_mistakes, black_inaccuracies,
                    white_blunders, white_mistakes, white_inaccuracies,
                ))

                # Insert move level data
                move_data = []
                for move in result["moves"]:
                    move_data.append((
                        game_id,
                        move["halfmove_count"],
                        str(move["move"]),
                        move["fen"],
                        move["turn"],
                        move["error"],
                        move["cp"],
                        move["mate"],
                        move["time_ratio"],
                        move["winning_chance"],
                        move["drawing_chance"],
                        move["losing_chance"],
                        move["is_check"],
                        move["is_checkmate"],
                    ))

                cursor.executemany("""
                INSERT INTO evaluation_move_level
                (game_id, halfmove_count, move, fen, turn, error, cp, mate,
                time_ratio, winning_chance, drawing_chance, losing_chance,
                is_check, is_checkmate)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                """, move_data)

            conn.commit()
            logger.info(f"Successfully saved {len(analysis_results)} games to database")

        except Exception as e:
            conn.rollback()
            logger.exception(f"Error saving to database: {e}")
        finally:
            conn.close()

    def _create_tf_feature(self, value):
        """Create appropriate TensorFlow feature from a value."""
        if isinstance(value, int):
            return tf.train.Feature(int64_list=tf.train.Int64List(value=[value]))
        if isinstance(value, float):
            return tf.train.Feature(float_list=tf.train.FloatList(value=[value]))
        if isinstance(value, str):
            return tf.train.Feature(bytes_list=tf.train.BytesList(value=[value.encode("utf-8")]))
        if isinstance(value, bytes):
            return tf.train.Feature(bytes_list=tf.train.BytesList(value=[value]))
        raise ValueError(f"Unsupported type: {type(value)}")

    def export_to_tfrecord(self, analysis_results: List[Dict[str, Any]], output_path: str) -> None:
        """Export analysis results to TFRecord format.

        Args:
            analysis_results (List[Dict]): List of analysis results from analyzed games
            output_path (str): Path to save the TFRecord file
        """
        logger.info(f"Exporting {len(analysis_results)} games to TFRecord at {output_path}")

        try:
            with tf.io.TFRecordWriter(output_path) as writer:
                for result in analysis_results:
                    if "error" in result:
                        continue

                    game_id = result["game_id"]
                    white_elo, black_elo = result["target"]

                    # Game level features
                    game_features = {
                        "estimated_time": self._create_tf_feature(result["estimated_time"] or 0),
                        "game_type": self._create_tf_feature(result["game_type"]),
                        "white_elo": self._create_tf_feature(white_elo),
                        "black_elo": self._create_tf_feature(black_elo),
                        "outcome": self._create_tf_feature(result["outcome"]),
                        "move_count": self._create_tf_feature(len(result["moves"])),
                        # Error counts
                        "white_blunders": self._create_tf_feature(result["white_errors"][0]),
                        "white_mistakes": self._create_tf_feature(result["white_errors"][1]),
                        "white_inaccuracies": self._create_tf_feature(result["white_errors"][2]),
                        "black_blunders": self._create_tf_feature(result["black_errors"][0]),
                        "black_mistakes": self._create_tf_feature(result["black_errors"][1]),
                        "black_inaccuracies": self._create_tf_feature(result["black_errors"][2]),
                    }

                    # TODO: Add move level features as needed
                    # This is a placeholder - implement the full encoding logic later

                    example = tf.train.Example(features=tf.train.Features(feature=game_features))
                    writer.write(example.SerializeToString())

            logger.info(f"Successfully exported to {output_path}")

        except Exception as e:
            logger.exception(f"Error exporting to TFRecord: {e}")

    def run_analysis(self, batch_size: int = 100, total_games: int = None,
                    game_type_filter: str = "rapid", start_offset: int = 0,
                    tfrecord_dir: str = "tfrecords") -> None:
        """Run analysis on unanalyzed games in parallel.

        Args:
            batch_size (int): Number of games to process in each batch
            total_games (int, optional): Total number of games to process
            game_type_filter (str): Type of games to analyze (rapid, blitz, classical, etc.)
            start_offset (int, optional): Initial offset for resuming a previous analysis run
            tfrecord_dir (str): Directory to save TFRecord files
        """
        games_processed = 0
        current_offset = start_offset
        batch_counter = 1

        # Create TFRecord directory if it doesn't exist
        os.makedirs(tfrecord_dir, exist_ok=True)

        logger.info(f"Starting analysis of {game_type_filter} games with offset {start_offset}...")

        while True:
            # Use a separate connection for fetching games since we'll be passing them to processes
            conn = sqlite3.connect(self.db_path)
            cursor = self.get_games_to_analyze(conn, limit=batch_size,
                                        game_type_filter=game_type_filter,
                                        offset=current_offset)

            games = cursor.fetchall()
            conn.close()

            if not games:
                logger.warning(f"No more {game_type_filter} games to analyze")
                break

            if total_games and games_processed >= total_games:
                logger.warning(f"Reached target of {total_games} games")
                break

            games_count = len(games)
            logger.debug(f"Processing batch of {games_count} {game_type_filter} games (offset: {current_offset})...")

            # Process games in parallel using ProcessPoolExecutor for CPU-bound tasks
            results = []
            with concurrent.futures.ProcessPoolExecutor(max_workers=self.max_workers) as executor:
                # We need to provide all necessary instance variables to analyze_game
                # since it will be executed in a separate process
                future_to_game = {
                    executor.submit(
                        self.analyze_game,
                        game,
                    ): game for game in games
                }

                for future in tqdm(concurrent.futures.as_completed(future_to_game), total=len(games)):
                    try:
                        game_result = future.result()
                        results.append(game_result)

                        # Log progress periodically
                        if len(results) % 10 == 0:
                            logger.info(f"Analyzed {len(results)}/{games_count} games in current batch")
                    except Exception:
                        game_id = future_to_game[future][0]  # First element is typically game_id
                        logger.exception(f"Game {game_id} generated an exception.")

            # Save results to database
            logger.info(f"Saving batch {batch_counter} results to database...")
            self.save_to_database(results)

            # Export to TFRecord
            tfrecord_path = os.path.join(tfrecord_dir, f"{game_type_filter}_batch_{batch_counter}.tfrecord")
            logger.info(f"Exporting batch {batch_counter} to TFRecord...")
            self.export_to_tfrecord(results, tfrecord_path)

            # Update counters
            games_processed += games_count
            current_offset += games_count
            batch_counter += 1

            logger.info(f"Completed batch {batch_counter-1}. Total games processed: {games_processed}")

        logger.info(f"Analysis complete. Processed {games_processed} games.")

In [ ]:
analyzer = ChessAnalyzer(Path("/home/vandy/work/chess/data/db.ocgdb.db3"), Path("/home/vandy/.local/bin/stockfish"), depth=16, threads=8, max_workers=8)

In [ ]:
con = sqlite3.connect("/home/vandy/work/chess/data/db.ocgdb.db3")
cur = con.cursor()

cur.execute("SELECT * FROM game_with_type limit 1")
game = cur.fetchall()
# game = list(game[0])
# game[14] = "1. e4 { [%eval 0.18] [%clk 0:01:00] } 1... c5 { [%eval 0.25] [%clk 0:01:00] } { B20 Sicilian Defense } 2. d3 { [%eval -0.07] [%clk 0:01:00] } 2... e6 { [%eval 0.0] [%clk 0:00:55] } 3. f3 { [%eval -0.39] [%clk 0:01:00] } 3... Nc6 { [%eval -0.36] [%clk 0:00:52] } 4. c3 { [%eval -0.43] [%clk 0:01:00] } 4... Nf6 { [%eval -0.3] [%clk 0:00:51] } 5. d4 { [%eval -0.56] [%clk 0:00:59] } 5... Be7?? { (-0.56 → 1.42) Blunder. cxd4 was best. } { [%eval 1.42] [%clk 0:00:51] } (5... cxd4 6. cxd4 Qb6 7. Ne2 d5 8. e5 Nd7 9. Nbc3 Be7 10. a3) 6. Bg5?? { (1.42 → -1.69) Blunder. d5 was best. } { [%eval -1.69] [%clk 0:00:57] } (6. d5 exd5 7. exd5 O-O 8. dxc6 Re8 9. Na3 bxc6 10. Nc2 d5 11. Kf2 Rb8) 6... h6? { (-1.69 → -0.50) Mistake. Qb6 was best. } { [%eval -0.5] [%clk 0:00:49] } (6... Qb6 7. Ne2 d5 8. e5 Qxb2 9. Nd2 Ng8 10. Bf4 cxd4 11. Rb1 Qa3 12. cxd4) 7. Bxf6?! { (-0.50 → -1.32) Inaccuracy. Be3 was best. } { [%eval -1.32] [%clk 0:00:57] } (7. Be3 O-O) 7... Bxf6 { [%eval -1.32] [%clk 0:00:49] } 8. dxc5 { [%eval -1.49] [%clk 0:00:56] } 8... O-O { [%eval -1.36] [%clk 0:00:47] } 9. Bb5 { [%eval -1.73] [%clk 0:00:54] } 9... Qa5?! { (-1.73 → -1.02) Inaccuracy. Be7 was best. } { [%eval -1.02] [%clk 0:00:46] } (9... Be7 10. Nd2 Bxc5 11. Qe2 a6 12. Bd3 b5 13. e5 b4 14. Qe4 g6 15. Ne2) 10. Bxc6?! { (-1.02 → -1.85) Inaccuracy. Qa4 was best. } { [%eval -1.85] [%clk 0:00:53] } (10. Qa4) 10... bxc6 { [%eval -1.78] [%clk 0:00:44] } 11. Ne2?! { (-1.78 → -2.97) Inaccuracy. b4 was best. } { [%eval -2.97] [%clk 0:00:49] } (11. b4 Qc7 12. Qd6 Qxd6 13. cxd6 a5 14. bxa5 Rxa5 15. f4 g5 16. Nf3 gxf4) 11... Qxc5 { [%eval -2.97] [%clk 0:00:44] } 12. b4 { [%eval -3.27] [%clk 0:00:45] } 12... Qe7 { [%eval -2.87] [%clk 0:00:38] } 13. O-O { [%eval -2.88] [%clk 0:00:44] } 13... Rd8 { [%eval -2.33] [%clk 0:00:38] } 14. Nf4?! { (-2.33 → -3.30) Inaccuracy. f4 was best. } { [%eval -3.3] [%clk 0:00:40] } (14. f4 e5 15. Nd2 Ba6 16. Re1 exf4 17. Nd4 Bxd4+ 18. cxd4 Qxb4 19. Nf3 c5) 14... d5 { [%eval -3.66] [%clk 0:00:36] } 15. exd5 { [%eval -3.3] [%clk 0:00:30] } 15... cxd5 { [%eval -3.13] [%clk 0:00:36] } 16. Nd2?! { (-3.13 → -4.89) Inaccuracy. Nh5 was best. } { [%eval -4.89] [%clk 0:00:28] } (16. Nh5 Be5 17. f4 Bc7 18. Nd2 g6 19. Qg4 Ba6 20. Rfe1 Kh7 21. Ng3 Qf6) 16... Bxc3 { [%eval -4.89] [%clk 0:00:34] } 17. Rc1 { [%eval -4.65] [%clk 0:00:25] } 17... Bxb4 { [%eval -4.55] [%clk 0:00:32] } 18. Rb1 { [%eval -5.53] [%clk 0:00:22] } 18... Bd6 { [%eval -4.94] [%clk 0:00:31] } 19. Rb3?! { (-4.94 → -7.01) Inaccuracy. g3 was best. } { [%eval -7.01] [%clk 0:00:18] } (19. g3 Qf6 20. Nb3 Ba6 21. Re1 Rac8 22. Qd2 Qc3 23. Nxe6 fxe6 24. Qxc3 Rxc3) 19... Bxf4 { [%eval -6.97] [%clk 0:00:29] } 20. g3 { [%eval -7.18] [%clk 0:00:17] } 20... Bxd2 { [%eval -6.71] [%clk 0:00:26] } 21. Qxd2 { [%eval -6.89] [%clk 0:00:16] } 21... Ba6 { [%eval -6.48] [%clk 0:00:25] } 22. Re1 { [%eval -6.88] [%clk 0:00:14] } 22... Rab8 { [%eval -6.09] [%clk 0:00:24] } 23. Rxb8 { [%eval -6.24] [%clk 0:00:12] } 23... Rxb8 { [%eval -5.78] [%clk 0:00:24] } 24. Qa5 { [%eval -6.29] [%clk 0:00:09] } 24... Bc4 { [%eval -6.38] [%clk 0:00:20] } 25. Qa4?! { (-6.38 → -9.26) Inaccuracy. h4 was best. } { [%eval -9.26] [%clk 0:00:07] } (25. h4 e5 26. a3 h5 27. Rc1 Qb7 28. Qc3 f6 29. Kg2 Qb2+ 30. Rc2 Qxc3) 25... Rc8 { [%eval -6.65] [%clk 0:00:18] } 26. Qa5 { [%eval -6.85] [%clk 0:00:04] } 26... Qc5+ { [%eval -6.8] [%clk 0:00:17] } 27. Qxc5 { [%eval -6.9] [%clk 0:00:03] } 27... Rxc5 { [%eval -6.74] [%clk 0:00:17] } 28. Rd1 { [%eval -7.3] [%clk 0:00:02] } 28... Ra5 { [%eval -7.18] [%clk 0:00:16] } 29. Kg2 { [%eval -7.45] [%clk 0:00:02] } 29... Rxa2+ { [%eval -7.42] [%clk 0:00:15] } 30. Kh3 { [%eval -7.46] [%clk 0:00:01] } 30... a5 { [%eval -7.49] [%clk 0:00:15] } 31. Rb1 { [%eval -7.25] [%clk 0:00:00] } 31... a4 { [%eval -6.82] [%clk 0:00:15] } { Black wins on time. } 0-1"
# game = tuple(game)
game

In [ ]:
# analyzer.analyze_game(game[0])

In [ ]:
# %%timeit
engine = chess.engine.SimpleEngine.popen_uci("/home/vandy/.local/bin/stockfish")
engine.configure({
    "Threads": 8,
    # "Hash": 2048,  # 2GB hash
    "Skill Level": 20,  # Max strength
    "UCI_LimitStrength": False,
    "UCI_ShowWDL": True,
})

test_fens = {
    "White winning": "6k1/pp3ppp/4p3/2P5/1P2P3/P3q3/5PPP/3R2K1 w - - 0 1",
    "Black winning": "1r3rk1/1p1b1ppp/p2p4/4p3/4P3/1P6/P1P1nPPP/6K1 w - - 0 1",
    "Test1": "1r4k1/p3qpp1/4p2p/3p4/Q1b5/5PP1/P6P/4R1K1 b - - 3 25",
}

board = chess.Board()

for name, fen in test_fens.items():
    board.set_fen(fen)
    info = engine.analyse(board, chess.engine.Limit(depth=16))
    print(f"{name}: {info['score'].white().score(mate_score=100000)/100:.2f}")



In [ ]:
games_cursor = analyzer.get_games_to_analyze(1, offset=50000)
game = games_cursor.fetchall()
game =  list(game[0])
game[14] = "1. e4 c5 2. d4 e6 { C00 French Defense: Franco-Sicilian Defense } 3. dxc5 Bxc5 4. Nf3 Nc6 5. Nc3 Nf6 6. Bd3 O-O 7. Be3 Bxe3 8. fxe3 h6 9. Qe2 d5 10. e5 Nd7 11. O-O-O Ndxe5 12. e4 Nxd3+ 13. Rxd3 d4 14. Nxd4 Nxd4 15. Qd2 e5 16. Ne2 Re8 17. h3 Bd7 18. Re1 Bc6 19. Nxd4 exd4 20. Rxd4 Qe7 21. Re2 Rad8 22. Rxd8 Rxd8 23. Qe3 Re8 24. Kd1 Qxe4 25. Qd2 Qxe2+ 26. Qxe2 Rxe2 27. Kxe2 Bxg2 28. Kf2 Bxh3 29. Kg3 Be6 30. b3 Kf8 31. Kf4 Ke7 32. Ke4 Kd6 33. Kd4 h5 34. Ke3 g5 35. Kf3 h4 36. Kg2 f5 37. c4 f4 38. a3 Kc5 39. b4+ Kxc4 40. b5 Kxb5 41. a4+ Kxa4 42. Kf3 b5 43. Ke4 b4 44. Kd3 b3 45. Kc3 Ka3 46. Kd4 b2 47. Ke5 b1=Q 48. Kxe6 Qb6+ 49. Kf5 Qb5+ 50. Kg4 a5 51. Kh5 Kb3 52. Kg4 Kc4 53. Kh5 Qe8+ 54. Kg4 Qe6+ 55. Kh5 a4 56. Kxg5 a3 57. Kxf4 a2 58. Kg5 a1=Q 59. Kxh4 Qa5 60. Kg3 Kc5 61. Kf3 Qa3+ 62. Kf4 Qc4+ 63. Ke5 Qad3 64. Kf6 Qce4 65. Kg5 Qdd5+ 66. Kf6 Qee6+ 67. Kg7 Qdd7+ 68. Kf8 Qee8# { Black wins by checkmate. } 0-1"

In [ ]:
game

In [ ]:
result = analyzer.analyze_game(tuple(game))

In [ ]:
result

In [ ]:
pgn_text = game[0][14] # PGN Text
time_control = game[0][10] # 180+0
base_time, increment, estimated_time = analyzer.parse_time_control(time_control)
pgn_io = io.StringIO(pgn_text)
game_parsed = chess.pgn.read_game(pgn_io)

In [ ]:
# Analyze the game
board = game_parsed.board()

move_features = []

white_blunders = 0
white_mistakes = 0
white_inaccuracies = 0
black_blunders = 0
black_mistakes = 0
black_inaccuracies = 0

In [ ]:
curr_node = game_parsed
halfmove_count = -1
turn = int(chess.BLACK)
prev_node = chess.pgn.Game()
prev_prev_clock = None

while curr_node:

    error = None
    move = curr_node.move

    if curr_node.parent is not None:
        board.push(move)

    info = engine.analyse(board, chess.engine.Limit(depth=16), info=chess.engine.Info.ALL)

    # position_fen = board.fen()
    score = info["score"].white()
    cp = score.score(mate_score=1000)
    mate = score.mate()
    time_spent = 0 if (not curr_node.parent) or (not prev_prev_clock) else prev_prev_clock - curr_node.clock()
    time_ratio = 0 if (not prev_node.parent) or (not prev_prev_clock) else time_spent / (prev_prev_clock + increment)
    halfmove_count += 1
    turn = int(not turn)
    is_check = int(board.is_check())
    is_checkmate = int(board.is_checkmate())

    try:
        curr_node.wdl = info["wdl"]
    except KeyError:
        wdl = chess.engine.Wdl(1000, 0, 0)
        curr_node.wdl = chess.engine.PovWdl(wdl, not turn)

    if curr_node.parent:
        error = analyzer.evaluate_move(prev_node.wdl, curr_node.wdl)
        if error is Errors.BLUNDER:
            if turn is int(chess.BLACK):
                white_blunders += 1
            else:
                black_blunders += 1
        if error is Errors.MISTAKE:
            if turn is int(chess.BLACK):
                white_mistakes += 1
            else:
                black_mistakes += 1
        if error is Errors.INACCURACY:
            if turn is int(chess.BLACK):
                white_inaccuracies += 1
            else:
                black_inaccuracies += 1

    # Store the evaluation score in the node for later use
    features = {
                    "halfmove_count": halfmove_count,
                    "move": str(move),
                    "fen": board.fen(),
                    "turn": turn,
                    "error": error.value if error else Errors.NONE,
                    "cp": cp,
                    "mate": mate,
                    "time_ratio": time_ratio,
                    "winning_chance": curr_node.wdl.white().winning_chance(),
                    "drawing_chance": curr_node.wdl.white().drawing_chance(),
                    "losing_chance": curr_node.wdl.white().losing_chance(),
                    "is_check": is_check,
                    "is_checkmate": is_checkmate,
                }

    move_features.append(features)

    prev_prev_clock, prev_node, curr_node = prev_node.clock(), curr_node, curr_node.next()

In [ ]:
engine.quit()